# Preparation

Run `pip install -r requirements.txt`, then run this notebook.

# Initial set up

I'm importing packages and defining some globals for the notebook here. I set a random seed for reproducibility.

> **AI usage disclaimer:** I've come to embrace AI-assisted development, the work presented in this notebook has been partially generated using LLMs, however, I've supervised the entire process and ultimately decided which parts to change, update, and customize. This disclaimer is an indicator of my commitment to ethical, responsible, and transparent use of AI. All the writing (unless stated) has been done by me, as a demonstration of my understanding of the problem and the chosen approach; also, this project has been built gradually using multiple commits so it evidences my work.

> **Execution disclaimer:** I own an RTX 5090 GPU, which I was unable to use due to some issues with my OS, so I was limited to use a MacBook pro with an M1 Max and 32 GB of memory, meaning that CUDA behavior has not been tested. I use a smaller model for MPS/CPU and a larger one if CUDA is available.

In [ ]:
import json
import random
from pathlib import Path

import torch
from PIL import Image, ImageDraw, ImageFont

from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen3VLForConditionalGeneration,
)
from qwen_vl_utils import process_vision_info

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

from trl import SFTConfig, SFTTrainer

In [ ]:
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

IS_CUDA = torch.cuda.is_available()
IS_MPS = torch.backends.mps.is_available()

if IS_CUDA:
    DEVICE = "cuda"
    MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
elif IS_MPS:
    DEVICE = "mps"
    MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"
else:
    DEVICE = "cpu"
    MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"

OUTPUT_DIR = "qwen3-vl-health-table-extraction-lora"
print(f"Device: {DEVICE} | IS_CUDA={IS_CUDA} | Model: {MODEL_ID}")

# Overall description

Since we are dealing with complex (and variable) form layouts, we have to represent this data in a structured way. The most natural representation (with many benchmarks available as well) is with standard HTML tables, which allow us to make use of rowspan and colspan to merge cells and create fairly complex tables in a digital form.

**The HTML table** is wrapped in a JSON envelope that carries key/value fields (patient identifiers *redacted for training*, form date, form type):

**AI generated sample**

```json
{
  "form_type": "basic_metabolic_panel",
  "fields": {"specimen_date": "2026-03-14", "ordering_provider": "REDACTED"},
  "table_html": "<table><tr><th>Test</th><th colspan=\"2\">Reference Range</th><th>Result</th></tr>...</table>"
}
```

The model's job is: given the form image, emit exactly this JSON object as a string, while still getting HTML's structural expressiveness for the table body.

# Synthetic training data [AI generated helper]

In [ ]:
def _render_synthetic_lab_form(path: Path, seed: int) -> dict:
    """Render a crude 'lab panel' table into a high-resolution PNG and return
    the matching ground-truth record. This stands in for a real, scanned
    health-form image + a human- or vendor-labeled structured target.

    In production this function is replaced entirely: images come from your
    document ingestion pipeline (scanner/fax/EHR export), and targets come
    from human annotation (e.g., via Label Studio with an HTML-table export
    plugin) or from a trusted upstream structured source used to
    bootstrap weak labels.
    """
    rng = random.Random(seed)
    tests = [
        ("Sodium", "136", "145", "mmol/L"),
        ("Potassium", "3.5", "5.1", "mmol/L"),
        ("Chloride", "98", "107", "mmol/L"),
        ("Glucose", "70", "99", "mg/dL"),
        ("BUN", "7", "20", "mg/dL"),
        ("Creatinine", "0.6", "1.3", "mg/dL"),
    ]

    # High-resolution canvas: scanned health forms are frequently
    # 2000-3000px on the long edge. We deliberately render at high res so
    # the notebook exercises the same resolution regime as real scans.
    width, height = 1700, 2200
    img = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("/Library/Fonts/SnellRoundhand.ttc", 36)
        font_small = ImageFont.truetype("/Library/Fonts/SnellRoundhand.ttc", 28)
    except Exception:
        font = ImageFont.load_default()
        font_small = font

    draw.text((60, 60), "COMPREHENSIVE METABOLIC PANEL", fill="black", font=font)
    draw.text((60, 110), f"Specimen Date: 2026-0{rng.randint(1,9)}-{rng.randint(10,28)}", fill="black", font=font_small)

    top = 220
    row_h = 70
    col_x = [60, 500, 800, 1050, 1350]
    headers = ["Test", "Result", "Ref Low", "Ref High", "Unit"]
    for x, h in zip(col_x, headers):
        draw.text((x, top), h, fill="black", font=font)
    draw.line([(60, top + 50), (1640, top + 50)], fill="black", width=3)

    rows_html = []
    for i, (name, lo, hi, unit) in enumerate(tests):
        y = top + 70 + i * row_h
        result = str(round(rng.uniform(float(lo) * 0.85, float(hi) * 1.15), 1))
        row_vals = [name, result, lo, hi, unit]
        for x, v in zip(col_x, row_vals):
            draw.text((x, y), v, fill="black", font=font_small)
        rows_html.append(
            "<tr><td>{}</td><td>{}</td><td>{}</td><td>{}</td><td>{}</td></tr>".format(*row_vals)
        )

    img.save(path)

    table_html = (
        "<table>"
        "<tr><th>Test</th><th>Result</th><th>Ref Low</th><th>Ref High</th><th>Unit</th></tr>"
        + "".join(rows_html)
        + "</table>"
    )
    target = {
        "form_type": "basic_metabolic_panel",
        "fields": {"specimen_date": "REDACTED", "ordering_provider": "REDACTED"},
        "table_html": table_html,
    }
    return {"image": str(path), "target": target}


def build_synthetic_manifest(n_examples: int, out_dir: str) -> list[dict]:
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    manifest = []
    for i in range(n_examples):
        img_path = out_path / f"form_{i:04d}.png"
        record = _render_synthetic_lab_form(img_path, seed=i)
        manifest.append(record)
    return manifest


synthetic_manifest = build_synthetic_manifest(n_examples=24, out_dir="./synthetic_health_forms")
print(f"Built {len(synthetic_manifest)} synthetic (image, target) pairs.")
print(json.dumps(synthetic_manifest[0], indent=2)[:600], "...")


# Qwen3-VL chat template

Qwen3-VL is instruction-tuned and expects input as a **list of role-tagged messages**, where each message's `content` is itself a list of typed content blocks (`{"type": "text", "text": ...}` or `{"type": "image", "image": ...}`).

For training, **three** messages are constructed per example: System message (general instructions), user turn (image and brief description), system turn (processing output used as ground truth).

The training consists on masking the ground truth and comparing the output to compute loss.


In [ ]:
SYSTEM_PROMPT = (
    "You are a meticulous medical document structuring assistant. You are shown an image "
    "of a health form that contains one or more tables. Your job is to transcribe the form's "
    "table(s) and a small set of header fields into a single JSON object with exactly this shape:\n\n"
    '{"form_type": <string>, "fields": {"specimen_date": <string or null>, '
    '"ordering_provider": <string or null>}, "table_html": <string>}\n\n'
    "Rules:\n"
    "1. `table_html` must be a valid, minimal HTML <table> using <tr>/<th>/<td> and, when the form "
    "uses merged cells, `rowspan`/`colspan` attributes that faithfully reproduce the visual layout.\n"
    "2. Transcribe every visible row and column; do not summarize, reorder, or drop rows.\n"
    "3. If a field is not present on the form, use null rather than guessing.\n"
    "4. Output ONLY the JSON object. No markdown fences, no commentary.\n"
    "5. Never fabricate patient-identifying values; if a field is redacted or illegible, output null."
)

USER_INSTRUCTION = "Extract this form's structured data as JSON, per the schema and rules above."

# AI generated
def format_example(record: dict) -> list[dict]:
    """Convert one manifest record into the 3-turn chat-message structure
    Qwen3-VL's processor expects.

    `record["image"]` may be a filesystem path, a URL, or a PIL.Image — the
    `qwen_vl_utils.process_vision_info` helper accepts all three inside the
    `{"type": "image", "image": ...}` block, which is convenient: we don't
    have to eagerly load every image into memory just to build the message
    list.
    """
    target_str = json.dumps(record["target"], ensure_ascii=False)
    return [
        {
            "role": "system",
            "content": [{"type": "text", "text": SYSTEM_PROMPT}],
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": record["image"]},
                {"type": "text", "text": USER_INSTRUCTION},
            ],
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": target_str}],
        },
    ]


formatted_examples = [format_example(r) for r in synthetic_manifest]
print(formatted_examples[0][1])   # the user turn
print()
print(formatted_examples[0][2])   # the assistant turn


# Visual tokens budget

To account for high resolution images and prevent hallucinations caused by a model incapable of reading the image content we set a processor that automatically scales the image resolution to the nearest ViT grid:

MIN_PIXELS <= H*W <= MAX_PIXELS


In [ ]:
# Hyperparameter tuning could be beneficial here if in scope to determine the best max pixels value within budget, 2048 has been used as an arbitrarily high value.
MIN_PIXELS = 256 * 32 * 32
MAX_PIXELS = 2048 * 32 * 32

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)
print(processor)


# Batch input + mask tokens for inference [AI generated]

Tokenize the *prompt-only* rendering (system + user turn, with the generation prompt opened) separately from the *full* rendering (system + user + assistant turn), and mask everything up to the length of the prompt-only tokenization.

In [ ]:
IMAGE_TOKEN_STRINGS = ["<|vision_start|>", "<|vision_end|>", "<|image_pad|>"]

def _image_token_ids(proc) -> list[int | None]:
    ids = [proc.tokenizer.convert_tokens_to_ids(t) for t in IMAGE_TOKEN_STRINGS]
    resolved = [i for i in ids if i is not None and i >= 0]
    if len(resolved) != len(IMAGE_TOKEN_STRINGS):
        print(
            "WARNING: not all image special tokens resolved to an id "
            f"({resolved} from {IMAGE_TOKEN_STRINGS}). Inspect "
            "proc.tokenizer.special_tokens_map / additional_special_tokens "
            "before trusting the collator's loss masking."
        )
    return resolved


IMAGE_TOKEN_IDS = _image_token_ids(processor)
print("Image-related special token ids:", IMAGE_TOKEN_IDS)


def collate_fn(examples: list[list[dict]]) -> dict[str, torch.Tensor]:
    """Custom multimodal collator for trl.SFTTrainer.

    `examples` is a list of chat-message lists (system/user/assistant), i.e.
    exactly what `format_example()` produces. Each call receives one
    mini-batch worth of examples.
    """
    full_texts, prompt_texts, image_inputs = [], [], []

    for messages in examples:
        # Full rendering: system + user + assistant -> this is the sequence used for the actual training.
        full_texts.append(
            processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        )
        # Prompt-only rendering: system + user, with the generation prompt
        # opened -> used purely to measure how many leading tokens to mask.
        prompt_texts.append(
            processor.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True)
        )
        imgs, _ = process_vision_info(messages)
        image_inputs.append(imgs[0] if imgs else None)

    batch = processor(
        text=full_texts,
        images=image_inputs,
        return_tensors="pt",
        padding=True,
    )

    labels = batch["input_ids"].clone()

    # 1) Mask padding.
    labels[labels == processor.tokenizer.pad_token_id] = -100

    # 2) Mask image-placeholder / vision-boundary tokens -- these carry no
    #    linguistic content to predict.
    for image_token_id in IMAGE_TOKEN_IDS:
        labels[labels == image_token_id] = -100

    # 3) Mask the prompt portion (system + user turn) of every sequence, so
    #    loss is computed only on the assistant's structured-output tokens.
    #    We recompute the prompt length per-example with the SAME tokenizer
    #    call path (tokenize=True) so left-padding side / truncation cannot
    #    silently desync the boundary from the batch's actual input_ids.
    for i, prompt_text in enumerate(prompt_texts):
        prompt_len = len(processor.tokenizer(prompt_text, add_special_tokens=False).input_ids)
        labels[i, :prompt_len] = -100

    batch["labels"] = labels
    return batch


# Multi-table synthetic data [AI generated helper]

In [ ]:
def _render_synthetic_multi_table_form(path: Path, seed: int) -> dict:
    """Render a page containing TWO lab panels stacked vertically, and
    return one record per panel with that panel's exact pixel bounding box
    (in the *original, full-resolution* image) plus its structured target.

    This stands in for a real multi-section health form (e.g., a combined
    metabolic + hematology requisition) where more than one table shares a
    page. As in Section 1.2, in production these boxes come from your
    annotation tool, not from a rendering script -- what matters is that the
    training data preserves that mapping between a table's location and its
    transcription target.
    """
    rng = random.Random(seed)
    panels = [
        ("BASIC METABOLIC PANEL", "basic_metabolic_panel", [
            ("Sodium", "136", "145", "mmol/L"),
            ("Potassium", "3.5", "5.1", "mmol/L"),
            ("Chloride", "98", "107", "mmol/L"),
            ("Glucose", "70", "99", "mg/dL"),
        ]),
        ("COMPLETE BLOOD COUNT", "complete_blood_count", [
            ("WBC", "4.5", "11.0", "10^3/uL"),
            ("Hemoglobin", "13.5", "17.5", "g/dL"),
            ("Platelets", "150", "400", "10^3/uL"),
        ]),
    ]

    width, height = 1700, 2600
    img = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("/Library/Fonts/SnellRoundhand.ttc", 36)
        font_small = ImageFont.truetype("/Library/Fonts/SnellRoundhand.ttc", 28)
    except Exception:
        font = ImageFont.load_default()
        font_small = font

    col_x = [60, 500, 800, 1050, 1350]
    headers = ["Test", "Result", "Ref Low", "Ref High", "Unit"]
    row_h = 70
    regions = []
    y_cursor = 60

    for title, form_type, tests in panels:
        panel_top = y_cursor
        draw.text((60, y_cursor), title, fill="black", font=font)
        y_cursor += 50
        draw.text((60, y_cursor), f"Specimen Date: 2026-0{rng.randint(1,9)}-{rng.randint(10,28)}", fill="black", font=font_small)
        y_cursor += 60

        header_top = y_cursor
        for x, h in zip(col_x, headers):
            draw.text((x, y_cursor), h, fill="black", font=font)
        draw.line([(60, y_cursor + 50), (1640, y_cursor + 50)], fill="black", width=3)
        y_cursor += 70

        rows_html = []
        for name, lo, hi, unit in tests:
            result = str(round(rng.uniform(float(lo) * 0.85, float(hi) * 1.15), 1))
            row_vals = [name, result, lo, hi, unit]
            for x, v in zip(col_x, row_vals):
                draw.text((x, y_cursor), v, fill="black", font=font_small)
            rows_html.append(
                "<tr><td>{}</td><td>{}</td><td>{}</td><td>{}</td><td>{}</td></tr>".format(*row_vals)
            )
            y_cursor += row_h

        panel_bottom = y_cursor + 20
        # bbox convention: [x1, y1, x2, y2] in the ORIGINAL image's pixel
        # space, covering the panel's title through its last table row.
        regions.append({
            "bbox": [40, panel_top - 10, 1660, panel_bottom],
            "label": form_type,
            "target": {
                "form_type": form_type,
                "fields": {"specimen_date": "REDACTED", "ordering_provider": "REDACTED"},
                "table_html": (
                    "<table><tr><th>Test</th><th>Result</th><th>Ref Low</th><th>Ref High</th><th>Unit</th></tr>"
                    + "".join(rows_html) + "</table>"
                ),
            },
        })
        y_cursor = panel_bottom + 80  # gap before next panel

    img.save(path)
    return {"image": str(path), "regions": regions}


def build_multi_table_manifest(n_examples: int, out_dir: str) -> list[dict]:
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    manifest = []
    for i in range(n_examples):
        img_path = out_path / f"multi_form_{i:04d}.png"
        manifest.append(_render_synthetic_multi_table_form(img_path, seed=i))
    return manifest


multi_table_manifest = build_multi_table_manifest(n_examples=12, out_dir="./synthetic_multi_table_forms")
print(f"Built {len(multi_table_manifest)} multi-table pages, "
      f"{sum(len(r['regions']) for r in multi_table_manifest)} table regions total.")
print(json.dumps(multi_table_manifest[0]['regions'][0], indent=2)[:400], "...")


# Detection (table boundaries) and transcription examples set up

The model has been trained as a general purpose tool, so the objective is to "improve" its framing (table detection) and transcription (character recognition in cropped images) capabilities through the usage of a special set of examples.

## Detection examples

The image containing the tables is resized to a smaller pixel budget since this part of the training is for inducing some "shape detection" capabilities into the model.

The idea is for the model to be able to detect the table boundaries and pick a set of coordinates (x1, y1) and (x2, y2) to locate a table. These coordinates are normalized to the range [0,1000] which is relative to the image's Width and Height. This normalization is because Qwen3-VL uses a relative 1000×1000 coordinate grid for its 2D grounding and bounding box generation, regardless of the image’s original aspect ratio or absolute pixel dimensions.

## Transcription examples

The original image (full resolution) is cropped using the detected boundaries plus a small fractional margin (or the image boundaries if the margin goes beyond these).

The model should be able to scan the text in the cropped region and output the HTML-embedded JSON format defined earlier.

# Objective

Given these two specialized tasks, we can proceed with a special training loop, where we run the tasks, compute the loss, and fine-tune.

In [ ]:
DETECTION_SYSTEM_PROMPT = (
    "You are a document layout analysis assistant. You are shown an image of a health form page "
    "that may contain one or more tables. Locate every table on the page and output ONLY a JSON "
    "list, one entry per table, of the form "
    '{"bbox_2d": [x1, y1, x2, y2], "label": <short form-type slug>}. '
    "Coordinates are on a NORMALIZED 0-1000 scale relative to this image's width and height "
    "(NOT raw pixel values): x1,x2 in [0,1000] represent the fraction of the image WIDTH, "
    "y1,y2 in [0,1000] represent the fraction of the image HEIGHT. [x1, y1] is the top-left corner "
    "and [x2, y2] the bottom-right corner, and the box must fully contain the table's title and "
    "every row. Output nothing else."
)
DETECTION_INSTRUCTION = "Detect every table on this page and return their bounding boxes as specified."

# Coarser than the MIN_PIXELS/MAX_PIXELS used before since localization doesn't require much detail
# so this has a lower budget to use less resources
DETECTION_MIN_PIXELS = 256 * 32 * 32
DETECTION_MAX_PIXELS = 640 * 32 * 32


def _to_normalized_1000(x1, y1, x2, y2, orig_w, orig_h):
    """Convert an absolute-pixel bbox in the ORIGINAL image to Qwen3-VL's
    normalized 0-1000 scale."""
    return [
        round(x1 / orig_w * 1000),
        round(y1 / orig_h * 1000),
        round(x2 / orig_w * 1000),
        round(y2 / orig_h * 1000),
    ]


def build_detection_example(record: dict) -> list[dict]:
    img = Image.open(record["image"])
    orig_w, orig_h = img.size

    target_regions = []
    for region in record["regions"]:
        x1, y1, x2, y2 = region["bbox"]
        target_regions.append({
            "bbox_2d": _to_normalized_1000(x1, y1, x2, y2, orig_w, orig_h),
            "label": region["label"],
        })

    return [
        {"role": "system", "content": [{"type": "text", "text": DETECTION_SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "image", "image": record["image"]},
            {"type": "text", "text": DETECTION_INSTRUCTION},
        ]},
        {"role": "assistant", "content": [{"type": "text", "text": json.dumps(target_regions)}]},
    ]


CROP_MARGIN_FRAC = 0.03  # to avoid clipping a table's border/title


def crop_region(image: Image.Image, bbox: list[float], margin_frac: float = CROP_MARGIN_FRAC) -> Image.Image:
    W, H = image.size
    x1, y1, x2, y2 = bbox
    mx, my = (x2 - x1) * margin_frac, (y2 - y1) * margin_frac
    x1, y1 = max(0, x1 - mx), max(0, y1 - my)
    x2, y2 = min(W, x2 + mx), min(H, y2 + my)
    return image.crop((int(x1), int(y1), int(x2), int(y2)))


def build_transcription_example(record: dict, region: dict) -> list[dict]:
    img = Image.open(record["image"])
    cropped = crop_region(img, region["bbox"])
    target_str = json.dumps(region["target"], ensure_ascii=False)
    return [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "image", "image": cropped},
            {"type": "text", "text": USER_INSTRUCTION},
        ]},
        {"role": "assistant", "content": [{"type": "text", "text": target_str}]},
    ]


detection_examples = [build_detection_example(r) for r in multi_table_manifest]
transcription_examples = [
    build_transcription_example(r, region)
    for r in multi_table_manifest
    for region in r["regions"]
]
print(f"{len(detection_examples)} detection examples, {len(transcription_examples)} cropped transcription examples.")
print(detection_examples[0][2])  # the detection target for one page (bbox_2d values should now be in [0,1000])


# Training considerations

## Maximizing the model's performance

There's one important detail to consider: since the training for transcription is performed on cropped images, inference will perform better on cropped images as well. This is a limitation that resembles human assets: we have several specialized employees instead of a single employee that does his job in a mediocre way.

## Tiling

Also, in many cases te tables will fit this detect + zoom + process approach, however, there can be other cases where tables are simply too big to fit on a single crop. Essentially, this forces us to perform some sort of tiling or sliding-window across the table, while also preserving the intended table structure.

To guarantee the preservation of the structure, this tiling process should target row bands (avoiding splits mid-row or mid-column), and allowing some overlap to piece everything together while deduplicating (think of this as solving a jigsaw puzzle).


# Model + PEFT

Tensors are data structures that store numbers with a determined precision. Normally we encounter models in half precision (float16) or in bf16.

> `Qwen3-VL-8B-Instruct` has roughly 9B total parameters (8B-class language backbone + ViT + merger/DeepStack fusion layers). In bf16, weights alone are ~18GB. A **full** fine-tune with AdamW needs, per trainable parameter: 2 bytes (bf16 weight) + 2 bytes (bf16 gradient) + 8 bytes (fp32 first+second moment in Adam) = 12 bytes, i.e. ~108GB just for optimizer state and gradients on top of the 18GB of weights — before activations. That's multi-A100 territory before you've fine-tuned anything. [AI generated]

By leveraging Quantized Low-Rank Adaptation (QLoRA), we can reduce the amount of memory needed during inference, but we can also quantize the model and tune a new set of parameters while freezing these quantized parameters. Quantization reduces precision, but this precision sacrifice is minimal during inference, training is where precision is needed (due to the gradient-based optimization).

# BitsAndBytes

We target 4-bit quantization (a large model with a strong 4-bit quantization is preferred over small models with weaker quantizations).

For Qwen-VL models, the recommended quantization type is NF4 (Normal Float 4), as the community agrees that it has a better theoretical performance and higher precision for weights initialized from a normal distribution compared to standard FP4.

In [ ]:
if IS_CUDA:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, # to save more VRAM with relatively low quality cost
        bnb_4bit_compute_dtype=torch.bfloat16, # to compute using bf16
    )
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )
else:
    bnb_config = None
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )

# min_pixels/max_pixels is already contained in the processor
model.config.use_cache = False  # re-enabled for inference later
print(model.config.torch_dtype, next(model.parameters()).dtype)

# Gradient checkpointing

We trade reduced memory usage for increased taining times.

In [ ]:
if IS_CUDA:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# Choosing target modules

To efficiently and effectively tune relevant modules, the pattern is restricted to the language mdoel decoder's attention and MLP projections: `q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj`. This freezes the 'vision' and 'language' layers, keeping the reasoning and perception modules intact.

In [ ]:
def find_target_linear_layers(model, include_keywords, exclude_prefixes=("visual",)):
    found = set()
    for name, module in model.named_modules():
        if any(name.startswith(p) for p in exclude_prefixes):
            continue
        cls_name = module.__class__.__name__
        is_linear = cls_name in ("Linear", "Linear4bit", "Linear8bitLt")
        if is_linear and any(k in name for k in include_keywords):
            found.add(name.split(".")[-1])
    return sorted(found)


LANGUAGE_MODEL_PROJECTIONS = (
    "q_proj", "k_proj", "v_proj", "o_proj",       # attention
    "gate_proj", "up_proj", "down_proj",          # MLP
)

target_modules = find_target_linear_layers(model, LANGUAGE_MODEL_PROJECTIONS)
print("LoRA target modules:", target_modules)

vision_related = sorted({n.split(".")[0] for n, _ in model.named_modules() if "vis" in n.lower()})
print("Top-level module name fragments containing 'vis':", vision_related)


# LoraConfig

| Argument | Value                      | Rationale                                                                                                                                                                                                       |
|---|----------------------------|-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `r` (rank) | **16**                     | Rank 8 is the default, 16 was chosen because we are also teaching the model a fairly rigid *new output grammar*, which benefits from a bit more expressive capacity than, for example, adjusting response tone. |
| `lora_alpha` | **32**                     | A heuristic of `alpha = 2 * r` was chosen                                                                                                                                                                  |
| `lora_dropout` | **0.05**                   | Just a small arbitrary value for regularization.                                                                                                                                                                |
| `bias` | `"none"`                   | `"none"` is the standard choice                                                                                                                                                                                 |
| `target_modules` | discovered list from above | LLM attention + MLP projections; vision tower untouched.                                                                                                                                                        |
| `task_type` | `"CAUSAL_LM"`              | Qwen3-VL's language modeling head is a standard causal LM head, PEFT needs this to correctly wrap `forward`/`generate`.                                                                                         |

All of the above are hyperparameters. Given enough resources, these are all candidates for hyperparameter tuning. For a simple hyperparameter search, the standard GridSearch approach with Cross Validation would be appropriate, for a more in-depth I would lean towards Successive Halving since I've found it to be good for complex searches.

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=target_modules,
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


# Training

Putting it all together for training, we define our test (eval) and training sets.
Training is 85% of the examples.
Eval is 15% of the examples.

This choice was arbitrary.

In [ ]:
_inner_collate_fn = collate_fn

def collate_fn(examples):
    unwrapped = [ex["messages"] if isinstance(ex, dict) else ex for ex in examples]
    return _inner_collate_fn(unwrapped)

class ChatMessageDataset(torch.utils.data.Dataset):
    def __init__(self, examples: list[list[dict]]):
        self.examples = examples

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, idx: int) -> dict:
        return {"messages": self.examples[idx]}

all_examples = formatted_examples + detection_examples + transcription_examples
random.shuffle(all_examples)

n_eval = max(1, int(0.15 * len(all_examples)))
train_examples = all_examples[n_eval:]
eval_examples = all_examples[:n_eval]

train_dataset = ChatMessageDataset(train_examples)
eval_dataset = ChatMessageDataset(eval_examples)
print(f"train: {len(train_dataset)} examples, eval: {len(eval_dataset)} examples")
print(train_dataset[0])

The trainer config is mostly arbitrary and the configurations have been made by consulting Gemini. Hyperparameter tuning would be ideal here.

In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3, # Consider for hyperparameter tuning (also, maybe consider early stopping?)
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=IS_CUDA,
    learning_rate=2e-4, # Consider for hyperparameter tuning
    lr_scheduler_type="cosine", # Consider for hyperparameter tuning
    warmup_ratio=0.03, # Consider for hyperparameter tuning
    max_grad_norm=0.3, # Consider for hyperparameter tuning
    optim="paged_adamw_8bit" if IS_CUDA else "adamw_torch", # Consider for hyperparameter tuning

    bf16=True,
    tf32=IS_CUDA,  # TensorFloat32 has no MPS/CPU equivalent

    logging_steps=5,
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="steps",
    save_steps=20,
    save_total_limit=3,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    load_best_model_at_end=True,

    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},
    packing=False,

    report_to="none",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collate_fn,
)

In [ ]:
train_result = trainer.train() # run without checkpoints
# train_result = trainer.train(resume_from_checkpoint=True)  # to resume an interrupted run

trainer.save_model(training_args.output_dir)
processor.save_pretrained(training_args.output_dir)
print(train_result.metrics)

# Evaluation [AI generated]

TEDS parses both the predicted and reference HTML tables into trees and computes a normalized tree-edit distance between them, giving a score in `[0, 1]` where 1.0 is a perfect structural + content match. (https://arxiv.org/abs/2208.00385)

### TEDS — Tree-Edit-Distance-based Similarity

This is a simplified implementation that I generated. It computes an F1 score by computing the precision and recall, comparing the similarity between a reference and an inferred table.


In [ ]:
from html.parser import HTMLParser


class _SimpleTableParser(HTMLParser):
    """Extremely small HTML-table parser sufficient for our controlled
    output format (no nested tables, no attributes besides rowspan/colspan).
    """

    def __init__(self):
        super().__init__()
        self.rows = []
        self._current_row = None
        self._current_cell = None
        self._current_attrs = {}

    def handle_starttag(self, tag, attrs):
        attrs = dict(attrs)
        if tag == "tr":
            self._current_row = []
        elif tag in ("td", "th"):
            self._current_cell = []
            self._current_attrs = attrs

    def handle_data(self, data):
        if self._current_cell is not None:
            self._current_cell.append(data)

    def handle_endtag(self, tag):
        if tag in ("td", "th") and self._current_row is not None:
            text = "".join(self._current_cell).strip()
            rowspan = int(self._current_attrs.get("rowspan", 1))
            colspan = int(self._current_attrs.get("colspan", 1))
            self._current_row.append((text, rowspan, colspan))
            self._current_cell = None
        elif tag == "tr" and self._current_row is not None:
            self.rows.append(self._current_row)
            self._current_row = None


def html_table_to_grid(html: str) -> set[tuple[int, int, str]]:
    """Expand an HTML table (with rowspan/colspan) into a set of
    (row_index, col_index, cell_text) occupancy triples, so that two tables
    with different-but-equivalent markup for the same visual grid compare
    equal.
    """
    parser = _SimpleTableParser()
    parser.feed(html)
    occupied: dict[tuple[int, int], str] = {}
    for r, row in enumerate(parser.rows):
        c = 0
        for text, rowspan, colspan in row:
            while (r, c) in occupied:
                c += 1
            for dr in range(rowspan):
                for dc in range(colspan):
                    occupied[(r + dr, c + dc)] = text
            c += colspan
    return {(r, c, t) for (r, c), t in occupied.items()}


def table_cell_f1(pred_html: str, ref_html: str) -> float:
    try:
        pred_cells = html_table_to_grid(pred_html)
    except Exception:
        return 0.0
    ref_cells = html_table_to_grid(ref_html)
    if not ref_cells:
        return 0.0
    tp = len(pred_cells & ref_cells)
    precision = tp / len(pred_cells) if pred_cells else 0.0
    recall = tp / len(ref_cells)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def evaluate_prediction(pred_json_str: str, ref_record: dict) -> dict:
    """Score one prediction against its reference `target` dict. Returns a
    dict of sub-metrics so you can see *how* a prediction failed, not just a
    single blended number.
    """
    try:
        pred = json.loads(pred_json_str)
    except json.JSONDecodeError:
        return {"parse_ok": False, "cell_f1": 0.0, "form_type_correct": False, "fields_exact_match": False}

    ref = ref_record["target"]
    cell_f1 = table_cell_f1(pred.get("table_html", ""), ref.get("table_html", ""))
    form_type_correct = pred.get("form_type") == ref.get("form_type")
    fields_exact_match = pred.get("fields") == ref.get("fields")
    return {
        "parse_ok": True,
        "cell_f1": cell_f1,
        "form_type_correct": form_type_correct,
        "fields_exact_match": fields_exact_match,
    }


_demo_ref = "<table><tr><th>Test</th><th colspan=\"2\">Range</th></tr><tr><td>Na</td><td>136</td><td>145</td></tr></table>"
print("Self-test (identical tables) cell_f1 ==", table_cell_f1(_demo_ref, _demo_ref), "(expect 1.0)")


## Inference

> **Please note:** This pipeline is trained using synthetic data, so the metrics calculated here are not representative of real-world data performance.

In [ ]:
model.config.use_cache = True
model.eval()

@torch.inference_mode()
def generate_prediction(model, processor, image_path, max_new_tokens: int = 1024) -> str: # AI generated
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": USER_INSTRUCTION},
            ],
        },
    ]
    text_input = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)
    model_inputs = processor(text=[text_input], images=image_inputs, return_tensors="pt").to(model.device)

    generated_ids = model.generate(**model_inputs, max_new_tokens=max_new_tokens, do_sample=False)
    trimmed = [out[len(inp):] for inp, out in zip(model_inputs.input_ids, generated_ids)]
    return processor.batch_decode(trimmed, skip_special_tokens=True)[0]

results = []
for record in synthetic_manifest[:n_eval]:
    pred_str = generate_prediction(model, processor, record["image"])
    results.append(evaluate_prediction(pred_str, record))

parse_rate = sum(r["parse_ok"] for r in results) / len(results)
mean_cell_f1 = sum(r["cell_f1"] for r in results) / len(results)
form_type_acc = sum(r["form_type_correct"] for r in results) / len(results)

print(f"JSON parse rate:      {parse_rate:.2%}")
print(f"Mean table cell F1:   {mean_cell_f1:.3f}")
print(f"form_type accuracy:   {form_type_acc:.2%}")


## Full page inference pipeline [co-developed using AI]

In [ ]:
detection_processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=DETECTION_MIN_PIXELS,
    max_pixels=DETECTION_MAX_PIXELS,
)

def _rescale_bbox_to_original(bbox_2d: list[float], orig_w: int, orig_h: int) -> list[float]:
    x1, y1, x2, y2 = bbox_2d
    return [x1 / 1000 * orig_w, y1 / 1000 * orig_h, x2 / 1000 * orig_w, y2 / 1000 * orig_h]

@torch.inference_mode()
def run_detection_pass(model, image: Image.Image, max_new_tokens: int = 512) -> list[dict]:
    orig_w, orig_h = image.size

    messages = [
        {"role": "system", "content": [{"type": "text", "text": DETECTION_SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": DETECTION_INSTRUCTION},
        ]},
    ]
    text_input = detection_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)
    model_inputs = detection_processor(text=[text_input], images=image_inputs, return_tensors="pt").to(model.device)

    generated_ids = model.generate(**model_inputs, max_new_tokens=max_new_tokens, do_sample=False)
    trimmed = [out[len(inp):] for inp, out in zip(model_inputs.input_ids, generated_ids)]
    raw_text = detection_processor.batch_decode(trimmed, skip_special_tokens=True)[0]

    try:
        data = json.loads(raw_text)
        if isinstance(data, dict):
            data = [data]
    except json.JSONDecodeError:
        return []

    regions = []
    for item in data:
        bbox = item.get("bbox_2d")
        if not bbox or len(bbox) != 4:
            continue
        regions.append({
            "bbox": _rescale_bbox_to_original(bbox, orig_w, orig_h),
            "label": item.get("label", "table"),
        })
    return regions


@torch.inference_mode()
def extract_structured_data(model, image_path: str) -> dict:
    image = Image.open(image_path)
    regions = run_detection_pass(model, image)

    if not regions:
        # Fallback: no regions detected degrade to single-shot whole-page transcription
        pred_str = generate_prediction(model, processor, image_path)
        try:
            parsed = json.loads(pred_str)
        except json.JSONDecodeError:
            parsed = {"raw_output": pred_str}
        return {"regions": [{"label": parsed.get("form_type", "unknown"), "bbox": [0, 0, *image.size], **parsed}]}

    page_result = {"regions": []}
    for region in regions:
        crop = crop_region(image, region["bbox"])
        pred_str = generate_prediction(model, processor, crop)  # generate_prediction accepts a PIL.Image directly
        try:
            parsed = json.loads(pred_str)
        except json.JSONDecodeError:
            parsed = {"raw_output": pred_str}
        page_result["regions"].append({"label": region["label"], "bbox": region["bbox"], **parsed})
    return page_result


In [ ]:
demo_result = extract_structured_data(model, multi_table_manifest[0]["image"])
print(json.dumps(demo_result, indent=2)[:800], "...")

## Region detection inference evaluation [AI generated]

In [ ]:
def bbox_iou(a: list[float], b: list[float]) -> float:
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def evaluate_page(pred_page: dict, ref_record: dict, iou_threshold: float = 0.5) -> dict:
    pred_regions = pred_page["regions"]
    ref_regions = ref_record["regions"]

    # Greedy matching: for each reference region, take the best remaining
    # unmatched prediction above the IoU threshold.
    unmatched_preds = list(range(len(pred_regions)))
    matches = []
    for ref_idx, ref in enumerate(ref_regions):
        best_iou, best_pred_idx = 0.0, None
        for pred_idx in unmatched_preds:
            iou = bbox_iou(pred_regions[pred_idx]["bbox"], ref["bbox"])
            if iou > best_iou:
                best_iou, best_pred_idx = iou, pred_idx
        if best_pred_idx is not None and best_iou >= iou_threshold:
            matches.append((best_pred_idx, ref_idx, best_iou))
            unmatched_preds.remove(best_pred_idx)

    precision = len(matches) / len(pred_regions) if pred_regions else 0.0
    recall = len(matches) / len(ref_regions) if ref_regions else 0.0

    matched_cell_f1s = []
    for pred_idx, ref_idx, _ in matches:
        matched_cell_f1s.append(
            table_cell_f1(pred_regions[pred_idx].get("table_html", ""), ref_regions[ref_idx]["target"]["table_html"])
        )

    return {
        "region_precision": precision,
        "region_recall": recall,
        "n_matched": len(matches),
        "n_predicted": len(pred_regions),
        "n_reference": len(ref_regions),
        "mean_matched_cell_f1": sum(matched_cell_f1s) / len(matched_cell_f1s) if matched_cell_f1s else 0.0,
    }

page_eval = evaluate_page(demo_result, multi_table_manifest[0])
print(page_eval)

## Tiling and merging inference evaluation [AI generated]

In [ ]:
def split_into_row_bands(image: Image.Image, n_bands: int, overlap_frac: float = 0.12) -> list[Image.Image]:
    """Split `image` into `n_bands` horizontal bands with `overlap_frac` of
    each band's height shared with its neighbor
    """
    W, H = image.size
    band_h = H / (n_bands - (n_bands - 1) * overlap_frac)
    overlap_h = band_h * overlap_frac
    bands = []
    y = 0.0
    for i in range(n_bands):
        y0 = max(0, y - (overlap_h if i > 0 else 0))
        y1 = min(H, y + band_h)
        bands.append(image.crop((0, int(y0), W, int(y1))))
        y += band_h - overlap_h
    return bands


def merge_row_band_html(html_fragments: list[str]) -> str:
    """Concatenate row-band table fragments into one table, dropping rows at
    the start of each fragment (after the first) that duplicate rows at the
    tail of the previous fragment, these are the rows both bands captured
    because of the deliberate overlap.
    """
    all_rows: list[tuple] = []
    header_row = None
    for frag_idx, html in enumerate(html_fragments):
        parser = _SimpleTableParser()
        parser.feed(html)
        rows = parser.rows
        if not rows:
            continue
        if frag_idx == 0:
            header_row = rows[0]
            all_rows.extend(rows)
            continue

        body_rows = rows[1:] if rows[0] == header_row else rows  # drop a repeated header, if the model re-emitted one
        # Drop a leading run of rows that duplicates the tail of what we've
        # already accumulated (the overlap region), searching a small window.
        search_window = min(5, len(body_rows), len(all_rows))
        dedup_from = 0
        for k in range(search_window, 0, -1):
            if body_rows[:k] == all_rows[-k:]:
                dedup_from = k
                break
        all_rows.extend(body_rows[dedup_from:])

    def _row_to_html(row):
        cells = "".join(
            f'<td rowspan="{rs}" colspan="{cs}">{text}</td>' if (rs > 1 or cs > 1) else f"<td>{text}</td>"
            for text, rs, cs in row
        )
        return f"<tr>{cells}</tr>"

    return "<table>" + "".join(_row_to_html(r) for r in all_rows) + "</table>"


# Self-test: split one of our existing single-table forms into 2 overlapping
# row bands, transcribe each band independently, and confirm the merged
# result matches the un-tiled ground truth exactly.
_demo_record = synthetic_manifest[0]
_full_image = Image.open(_demo_record["image"])
_bands = split_into_row_bands(_full_image, n_bands=2, overlap_frac=0.15)
_band_predictions = [generate_prediction(model, processor, band) for band in _bands]
_band_html_fragments = [json.loads(p).get("table_html", "<table></table>") for p in _band_predictions]
_merged_html = merge_row_band_html(_band_html_fragments)

print("Row-band merge cell_f1 vs. ground truth:",
      table_cell_f1(_merged_html, _demo_record["target"]["table_html"]))


# Deployment

This final steps merges the LoRA deltas with the base model, saving a final model in `{OUTPUT_DIR}-merged`.

In [ ]:
from peft import PeftModel

base_model_bf16 = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
merged_model = PeftModel.from_pretrained(base_model_bf16, training_args.output_dir)
merged_model = merged_model.merge_and_unload()

merged_output_dir = f"{OUTPUT_DIR}-merged"
merged_model.save_pretrained(merged_output_dir, safe_serialization=True)
processor.save_pretrained(merged_output_dir)
print(f"Merged, deployable model written to: {merged_output_dir}")

# [EXTRA] Hyperparameter tuning [AI generated]

I generated this routine that can be used for performing a small hyperparameter tuning over the LoRA config. The configuration with highest F1 score is used.

In [ ]:
import gc
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer
from transformers import Qwen3VLForConditionalGeneration

LORA_SEARCH_SPACE = [
    {"r": 8,  "lora_alpha": 16, "lora_dropout": 0.05},
    {"r": 16, "lora_alpha": 32, "lora_dropout": 0.05},
    {"r": 32, "lora_alpha": 64, "lora_dropout": 0.05},
    {"r": 16, "lora_alpha": 32, "lora_dropout": 0.10},
] # multiple combinations can be created but this is costly

N_EVAL_EXAMPLES_FOR_CELL_F1 = 3
MAX_STEPS_PER_TRIAL = 40

def _free_memory():
    gc.collect()
    if IS_CUDA:
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()


def _fresh_base_model():
    """Reload the base model per trial so adapters never stack on a previous
    trial's LoRA weights (get_peft_model on an already-adapted model would
    silently compose them instead of giving you a clean comparison)."""
    kwargs = dict(torch_dtype=torch.bfloat16, device_map="auto")
    if IS_CUDA:
        kwargs["quantization_config"] = bnb_config
    m = Qwen3VLForConditionalGeneration.from_pretrained(MODEL_ID, **kwargs)
    m.config.use_cache = False
    return m


def run_trial(lora_kwargs: dict) -> dict:
    trial_model = _fresh_base_model()
    trial_config = LoraConfig(
        target_modules=target_modules,
        bias="none",
        task_type="CAUSAL_LM",
        **lora_kwargs,
    )
    trial_model = get_peft_model(trial_model, trial_config)

    trial_args = SFTConfig(
        output_dir=f"./lora_sweep/r{lora_kwargs['r']}_a{lora_kwargs['lora_alpha']}_d{lora_kwargs['lora_dropout']}",
        max_steps=MAX_STEPS_PER_TRIAL,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        gradient_checkpointing=True,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        max_grad_norm=0.3,
        optim="paged_adamw_8bit" if IS_CUDA else "adamw_torch",
        bf16=True,
        tf32=IS_CUDA,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=MAX_STEPS_PER_TRIAL,
        save_strategy="no",
        remove_unused_columns=False,
        dataset_kwargs={"skip_prepare_dataset": True},
        packing=False,
        report_to="none",
        seed=SEED,
    )

    trial_trainer = SFTTrainer(
        model=trial_model,
        args=trial_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=collate_fn,
    )
    trial_trainer.train()
    eval_loss = trial_trainer.evaluate()["eval_loss"]

    trial_model.eval()
    cell_f1s = []
    for record in synthetic_manifest[:N_EVAL_EXAMPLES_FOR_CELL_F1]:
        pred_str = generate_prediction(trial_model, processor, record["image"])
        cell_f1s.append(evaluate_prediction(pred_str, record)["cell_f1"])
    mean_cell_f1 = sum(cell_f1s) / len(cell_f1s)

    del trial_trainer, trial_model
    _free_memory()

    return {**lora_kwargs, "eval_loss": eval_loss, "cell_f1": mean_cell_f1}

In [ ]:
# results = [run_trial(cfg) for cfg in LORA_SEARCH_SPACE] # TODO: uncomment

# Rank by cell_f1 first (higher is better),
# eval_loss as a tie-breaker (lower is better).
# results_sorted = sorted(results, key=lambda r: (-r["cell_f1"], r["eval_loss"])) # TODO: uncomment

# print(f"{'r':>4} {'alpha':>6} {'dropout':>8} {'eval_loss':>10} {'cell_f1':>8}") # TODO: uncomment
# for r in results_sorted: # TODO: uncomment
#     print(f"{r['r']:>4} {r['lora_alpha']:>6} {r['lora_dropout']:>8} {r['eval_loss']:>10.4f} {r['cell_f1']:>8.3f}") # TODO: uncomment
#
# best = results_sorted[0] # TODO: uncomment
# print(f"\nBest config: r={best['r']}, lora_alpha={best['lora_alpha']}, lora_dropout={best['lora_dropout']} " # TODO: uncomment
#       f"(cell_f1={best['cell_f1']:.3f}, eval_loss={best['eval_loss']:.4f})") # TODO: uncomment

In [ ]:
# TODO: uncomment

# lora_config = LoraConfig(
#     r=best["r"],
#     lora_alpha=best["lora_alpha"],
#     lora_dropout=best["lora_dropout"],
#     bias="none",
#     target_modules=target_modules,
#     task_type="CAUSAL_LM",
# )

# model = get_peft_model(model, lora_config)
# model.print_trainable_parameters()